# Face Attribute Editing - SD1.5 vs SDXL Inference + Evaluation

This notebook is the two-model version. Playground v2.5 is intentionally excluded because its inpainting output was unstable/noisy for this task.

Expected Drive layout:

```text
/content/drive/MyDrive/face-attr-edit/
  processed/                         # or processed.zip
    manifests/test.jsonl
    images_512, images_768, masks_hard, masks_soft, ...
  exports/best_loras/
    sd15_faceattr_lora.safetensors
    sdxl_faceattr_lora.safetensors
  runs/face_attr_edit_v2_full_inpaint/checkpoints/attr_classifier/best.pt  # recommended for full metrics
```

Outputs include the normal evaluation tables plus pairwise visuals:

```text
original | sd15 best | sdxl best
```

The pairwise CSV also includes candidate paths, so you can later choose the best model/candidate manually.

In [ ]:
# 1. Install dependencies
%pip install -q "diffusers==0.31.0" "transformers>=4.46,<5" "accelerate>=1.1,<2" "peft>=0.13,<1" "safetensors>=0.4.5,<1" "huggingface_hub>=0.26,<1" "lpips==0.1.4" "opencv-python-headless>=4.12,<5" "pandas>=2.2.3,<2.4" "tqdm>=4.67,<5" "PyYAML>=6.0.3,<6.1" "tabulate>=0.9,<1"

In [ ]:
# 2. Mount Drive and configure paths
from pathlib import Path
import os, json, random, shutil, time, zipfile, gc, math
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('[INFO] Not running in Colab or Drive already mounted:', e)

SEED = 42
RUN_TAG = 'face_attr_edit_v2_full_inpaint'
DRIVE_PROJECT = Path('/content/drive/MyDrive/face-attr-edit')
RUN_DIR = DRIVE_PROJECT / 'runs' / RUN_TAG
EXPORTS_DIR = DRIVE_PROJECT / 'exports'
LORA_DIR = EXPORTS_DIR / 'best_loras'

# Final comparison set: Playground is excluded.
COMPARE_MODELS = ['sd15', 'sdxl']

# Inference set. With SKIP_EXISTING=True, you can leave both here safely:
# existing sd15/sdxl outputs are reused; missing ones are generated.
MODELS_TO_RUN = ['sd15', 'sdxl']

TASKS = ['add_eyeglasses', 'make_smiling', 'make_older']
SPLIT_NAME = 'test'
SAMPLES_PER_TASK = 12
NUM_CANDIDATES = 3
NUM_INFERENCE_STEPS = 30
LOW_VRAM = True  # Keep True for T4/16GB or weaker; it uses CPU offload.
SKIP_EXISTING = True

# Optional: paste a HF token if the runtime cannot download a base model.
HF_TOKEN = ''
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

os.environ.setdefault('HF_HOME', '/content/hf_cache')
os.environ.setdefault('HF_HUB_CACHE', '/content/hf_cache/hub')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

for d in [
    RUN_DIR, EXPORTS_DIR, LORA_DIR,
    RUN_DIR / 'edited', RUN_DIR / 'metrics', RUN_DIR / 'reports',
    RUN_DIR / 'reports' / 'qualitative', RUN_DIR / 'reports' / 'pairwise',
    EXPORTS_DIR / 'comparison_grids',
]:
    d.mkdir(parents=True, exist_ok=True)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('Drive project:', DRIVE_PROJECT)
print('Run dir:', RUN_DIR)
print('Compare models:', COMPARE_MODELS)
print('Models to run:', MODELS_TO_RUN)

In [ ]:
# 3. Locate processed data and LoRA/checkpoint files

def find_processed_dir() -> Path:
    candidates = [
        DRIVE_PROJECT / 'processed',
        DRIVE_PROJECT / 'process',
        Path('/content/drive/MyDrive/processed'),
        Path('/content/drive/MyDrive/process'),
        Path('/content/face-attr-edit/processed'),
    ]
    for p in candidates:
        if (p / 'manifests' / f'{SPLIT_NAME}.jsonl').exists():
            return p

    zip_candidates = [
        DRIVE_PROJECT / 'processed.zip',
        Path('/content/drive/MyDrive/processed.zip'),
        Path('/content/processed.zip'),
    ]
    for z in zip_candidates:
        if z.exists():
            out_root = DRIVE_PROJECT
            print('[UNZIP]', z, '->', out_root)
            with zipfile.ZipFile(z) as zipf:
                zipf.extractall(out_root)
            for p in candidates:
                if (p / 'manifests' / f'{SPLIT_NAME}.jsonl').exists():
                    return p
    raise FileNotFoundError('Cannot find processed data folder or processed.zip on Drive.')

PROCESSED_DIR = find_processed_dir()
MANIFEST_DIR = PROCESSED_DIR / 'manifests'

MODEL_CFG = {
    'sd15': {
        'base': 'runwayml/stable-diffusion-v1-5',
        'architecture': 'sd15',
        'resolution': 512,
        'guidance_scale': 7.5,
    },
    'sdxl': {
        'base': 'stabilityai/stable-diffusion-xl-base-1.0',
        'architecture': 'sdxl',
        'resolution': 768,
        'guidance_scale': 6.5,
    },
}

TARGET_PROMPTS = {
    'add_eyeglasses': 'a high quality realistic face portrait, wearing natural eyeglasses with clear frames and lenses, neutral expression, natural skin texture, sharp facial details',
    'make_smiling': 'a high quality realistic face portrait, smiling naturally with raised mouth corners, friendly expression, natural skin texture, sharp facial details',
    'make_older': 'a high quality realistic face portrait of an older adult, mature face, subtle age lines, natural older facial features, natural skin texture, sharp facial details',
}
NEGATIVE_PROMPTS = {
    'add_eyeglasses': 'deformed face, different person, identity change, extra eyes, bad eyes, distorted glasses, blurry, low quality, artifacts',
    'make_smiling': 'deformed mouth, extra teeth, distorted teeth, different person, identity change, blurry, low quality, artifacts',
    'make_older': 'different person, identity change, extreme wrinkles, zombie, caricature, deformed face, blurry, low quality, artifacts',
}
STRENGTH = {
    'add_eyeglasses': {'sd15': 0.86, 'sdxl': 0.84},
    'make_smiling': {'sd15': 0.78, 'sdxl': 0.76},
    'make_older': {'sd15': 0.88, 'sdxl': 0.86},
}
LORA_SCALE = {'sd15': 1.0, 'sdxl': 1.0}
TASK_TO_ATTR_INDEX = {'add_eyeglasses': 0, 'make_smiling': 1, 'make_older': 2}
ATTR_COLUMNS = ['Eyeglasses', 'Smiling', 'Young']
TASK_TO_HARD_MASK_KEY = {'add_eyeglasses': 'mask_eyeglasses_hard', 'make_smiling': 'mask_smile_hard', 'make_older': 'mask_age_hard'}
TASK_TO_SOFT_MASK_KEY = {'add_eyeglasses': 'mask_eyeglasses_soft', 'make_smiling': 'mask_smile_soft', 'make_older': 'mask_age_soft'}

print('Processed dir:', PROCESSED_DIR)
for model_id in COMPARE_MODELS:
    p = LORA_DIR / f'{model_id}_faceattr_lora.safetensors'
    print(model_id, 'LoRA:', p, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else None)

classifier_candidates = [
    RUN_DIR / 'checkpoints' / 'attr_classifier' / 'best.pt',
    DRIVE_PROJECT / 'checkpoints' / 'attr_classifier' / 'best.pt',
    DRIVE_PROJECT / 'best.pt',
]
CLASSIFIER_PATH = next((p for p in classifier_candidates if p.exists()), None)
print('Classifier:', CLASSIFIER_PATH)

In [ ]:
# 4. Manifest and metric helpers
import cv2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True


def resolve_processed_path(stored: str) -> str:
    p = Path(stored)
    if p.exists():
        return str(p)
    parts = list(p.parts)
    if 'processed' in parts:
        idx = parts.index('processed')
        local = PROCESSED_DIR / Path(*parts[idx + 1:])
        if local.exists():
            return str(local)
    local = PROCESSED_DIR / p.name
    if local.exists():
        return str(local)
    return str(p)


def localize_record(r: Dict[str, Any]) -> Dict[str, Any]:
    r = dict(r)
    for k, v in list(r.items()):
        if isinstance(v, str) and (k.startswith('image_') or k.startswith('mask_')):
            r[k] = resolve_processed_path(v)
    return r


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(localize_record(json.loads(line)))
    return rows


def take_n(records: List[Dict[str, Any]], n: int, seed: int) -> List[Dict[str, Any]]:
    records = list(records)
    rng = random.Random(seed)
    rng.shuffle(records)
    return records[:min(n, len(records))]


def background_l1(original: Image.Image, edited: Image.Image, mask: Image.Image) -> float:
    o = np.asarray(original.convert('RGB')).astype(np.float32) / 255.0
    e = np.asarray(edited.convert('RGB').resize(original.size, Image.Resampling.LANCZOS)).astype(np.float32) / 255.0
    m = np.asarray(mask.convert('L').resize(original.size, Image.Resampling.BILINEAR)).astype(np.float32) / 255.0
    bg = 1.0 - m[..., None]
    return float((np.abs(o - e) * bg).sum() / max(float(bg.sum()) * 3.0, 1e-6))


def background_ssim(original: Image.Image, edited: Image.Image, mask: Image.Image) -> float:
    o = np.asarray(original.convert('L')).astype(np.float32) / 255.0
    e = np.asarray(edited.convert('L').resize(original.size, Image.Resampling.LANCZOS)).astype(np.float32) / 255.0
    m = np.asarray(mask.convert('L').resize(original.size, Image.Resampling.BILINEAR)).astype(np.float32) / 255.0
    bg = 1.0 - m
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    mx = cv2.GaussianBlur(o, (11, 11), 1.5)
    my = cv2.GaussianBlur(e, (11, 11), 1.5)
    sx = cv2.GaussianBlur(o * o, (11, 11), 1.5) - mx * mx
    sy = cv2.GaussianBlur(e * e, (11, 11), 1.5) - my * my
    sxy = cv2.GaussianBlur(o * e, (11, 11), 1.5) - mx * my
    ssim = ((2 * mx * my + c1) * (2 * sxy + c2)) / ((mx * mx + my * my + c1) * (sx + sy + c2) + 1e-8)
    return float((ssim * bg).sum() / max(float(bg.sum()), 1e-6))

_LPIPS_MODEL = None

def lpips_distance(a_img: Image.Image, b_img: Image.Image):
    global _LPIPS_MODEL
    try:
        import lpips
        if _LPIPS_MODEL is None:
            dev = 'cuda' if torch.cuda.is_available() else 'cpu'
            _LPIPS_MODEL = lpips.LPIPS(net='alex').to(dev).eval()
        a = np.array(a_img.convert('RGB').resize((256, 256))).astype(np.float32) / 127.5 - 1
        b = np.array(b_img.convert('RGB').resize((256, 256))).astype(np.float32) / 127.5 - 1
        ta = torch.from_numpy(a).permute(2, 0, 1).unsqueeze(0).to(next(_LPIPS_MODEL.parameters()).device)
        tb = torch.from_numpy(b).permute(2, 0, 1).unsqueeze(0).to(next(_LPIPS_MODEL.parameters()).device)
        with torch.no_grad():
            return float(_LPIPS_MODEL(ta, tb).item())
    except Exception as e:
        print('[WARN] LPIPS unavailable:', e)
        return None


def blend_with_soft_mask(original: Image.Image, edited: Image.Image, mask: Image.Image) -> Image.Image:
    return Image.composite(
        edited.convert('RGB').resize(original.size, Image.Resampling.LANCZOS),
        original.convert('RGB'),
        mask.convert('L').resize(original.size, Image.Resampling.BILINEAR),
    )

records = load_jsonl(MANIFEST_DIR / f'{SPLIT_NAME}.jsonl')
print('Loaded records:', len(records), 'from', MANIFEST_DIR / f'{SPLIT_NAME}.jsonl')
print('Example image exists:', Path(records[0]['image_512']).exists() if records else None)

In [ ]:
# 5. Pipeline loading and batch editing
from diffusers import AutoPipelineForInpainting


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def load_inpaint_pipeline(model_id: str):
    cfg = MODEL_CFG[model_id]
    base = cfg['base']
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    kwargs = dict(torch_dtype=dtype, use_safetensors=True)
    try:
        pipe = AutoPipelineForInpainting.from_pretrained(base, variant='fp16', **kwargs)
    except Exception as e:
        print('[INFO] Retry without fp16 variant:', e)
        pipe = AutoPipelineForInpainting.from_pretrained(base, **kwargs)

    lora_path = LORA_DIR / f'{model_id}_faceattr_lora.safetensors'
    if not lora_path.exists():
        raise FileNotFoundError(f'Missing LoRA: {lora_path}')
    try:
        pipe.load_lora_weights(str(lora_path), adapter_name=model_id)
        scale = float(LORA_SCALE.get(model_id, 1.0))
        pipe.set_adapters([model_id], adapter_weights=[scale])
        print(f'[OK] loaded LoRA: {lora_path} scale={scale}')
    except TypeError:
        pipe.load_lora_weights(str(lora_path))
        print('[OK] loaded LoRA:', lora_path)

    pipe.enable_attention_slicing()
    try:
        pipe.enable_vae_slicing()
        pipe.enable_vae_tiling()
    except Exception:
        pass

    if torch.cuda.is_available():
        if LOW_VRAM:
            pipe.enable_model_cpu_offload()
            print('[LOW VRAM] enable_model_cpu_offload active')
        else:
            pipe.to('cuda')
    else:
        pipe.to('cpu')
    return pipe


def score_candidate(c: Dict[str, Any]) -> float:
    bg = 1.0 - min(float(c.get('background_l1', 1.0)) / 0.08, 1.0)
    lp = c.get('lpips')
    lp_score = 0.5 if lp is None or (isinstance(lp, float) and math.isnan(lp)) else 1.0 - min(float(lp) / 0.35, 1.0)
    # Attribute and identity scoring happen in evaluation. Candidate selection uses preservation metrics here.
    return float(0.50 * bg + 0.50 * lp_score)


def planned_records_for_model(model_id: str):
    root = RUN_DIR / 'edited' / model_id
    planned = {}
    missing_meta = []
    existing_meta = []
    for task in TASKS:
        task_recs = take_n([r for r in records if task in r.get('eligible_tasks', [])], SAMPLES_PER_TASK, SEED + len(task))
        planned[task] = task_recs
        task_dir = root / task
        for r in task_recs:
            meta_path = task_dir / f"{str(r['id'])}_meta.json"
            if meta_path.exists():
                existing_meta.append(meta_path)
            else:
                missing_meta.append(meta_path)
    return planned, existing_meta, missing_meta


def batch_edit_model(model_id: str):
    cfg = MODEL_CFG[model_id]
    res = int(cfg['resolution'])
    root = RUN_DIR / 'edited' / model_id
    root.mkdir(parents=True, exist_ok=True)

    planned, existing_meta, missing_meta = planned_records_for_model(model_id)
    if SKIP_EXISTING and not missing_meta:
        print(f'[SKIP MODEL] {model_id}: all expected {len(existing_meta)} meta files already exist; no pipeline load needed.')
        return [json.loads(p.read_text(encoding='utf-8')) for p in existing_meta]

    if SKIP_EXISTING:
        print(f'[RESUME MODEL] {model_id}: {len(existing_meta)} existing, {len(missing_meta)} missing.')

    pipe = load_inpaint_pipeline(model_id)
    all_meta = []

    for task, task_recs in planned.items():
        task_dir = root / task
        task_dir.mkdir(parents=True, exist_ok=True)
        print(f'[EDIT] {model_id} {task}: {len(task_recs)} images')

        for r in tqdm(task_recs, desc=f'{model_id}/{task}'):
            image_id = str(r['id'])
            meta_path = task_dir / f'{image_id}_meta.json'
            if SKIP_EXISTING and meta_path.exists():
                all_meta.append(json.loads(meta_path.read_text(encoding='utf-8')))
                continue

            orig = Image.open(r[f'image_{res}']).convert('RGB')
            hard = Image.open(r[TASK_TO_HARD_MASK_KEY[task]]).convert('L').resize((res, res), Image.Resampling.NEAREST)
            soft = Image.open(r[TASK_TO_SOFT_MASK_KEY[task]]).convert('L').resize((res, res), Image.Resampling.BILINEAR)

            candidates = []
            for ci in range(NUM_CANDIDATES):
                seed = SEED + ci * 1000 + int(image_id)
                gen = torch.Generator(device='cpu').manual_seed(seed)
                with torch.inference_mode():
                    result = pipe(
                        prompt=TARGET_PROMPTS[task],
                        negative_prompt=NEGATIVE_PROMPTS[task],
                        image=orig,
                        mask_image=hard,
                        strength=STRENGTH[task][model_id],
                        guidance_scale=cfg['guidance_scale'],
                        num_inference_steps=NUM_INFERENCE_STEPS,
                        generator=gen,
                    ).images[0]
                blend = blend_with_soft_mask(orig, result, soft)
                raw_path = task_dir / f'{image_id}_candidate{ci}_raw.jpg'
                blend_path = task_dir / f'{image_id}_candidate{ci}_blend.jpg'
                result.save(raw_path, quality=95)
                blend.save(blend_path, quality=95)
                candidates.append({
                    'candidate_id': ci,
                    'seed': seed,
                    'raw_path': str(raw_path),
                    'blend_path': str(blend_path),
                    'path': str(blend_path),
                    'background_l1': background_l1(orig, blend, soft),
                    'lpips': lpips_distance(orig, blend),
                    'identity_similarity': None,
                })

            for c in candidates:
                c['selection_score'] = score_candidate(c)
            best_idx = int(np.argmax([c['selection_score'] for c in candidates]))
            best_path = task_dir / f'{image_id}_best.jpg'
            Image.open(candidates[best_idx]['blend_path']).save(best_path, quality=95)

            meta = {
                'id': image_id,
                'model_id': model_id,
                'task': task,
                'engine': 'inpainting',
                'source_image': r[f'image_{res}'],
                'best_path': str(best_path),
                'mask_soft': r[TASK_TO_SOFT_MASK_KEY[task]],
                'mask_hard': r[TASK_TO_HARD_MASK_KEY[task]],
                'strength': STRENGTH[task][model_id],
                'guidance_scale': cfg['guidance_scale'],
                'num_inference_steps': NUM_INFERENCE_STEPS,
                'selected_candidate': best_idx,
                'selection_score': candidates[best_idx]['selection_score'],
                'candidates': candidates,
            }
            meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')
            all_meta.append(meta)

    del pipe
    cleanup()
    return all_meta

for model_id in MODELS_TO_RUN:
    batch_edit_model(model_id)

In [ ]:
# 6. Attribute classifier helpers for evaluation
import torch.nn as nn
from torchvision import transforms
from torchvision.models import efficientnet_b0

CLASSIFIER_IMAGE_SIZE = 224
CLASSIFIER_THRESHOLD = 0.5


def build_attr_classifier() -> nn.Module:
    model = efficientnet_b0(weights=None)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(ATTR_COLUMNS))
    return model


def load_classifier(path: Optional[Path]):
    if path is None or not Path(path).exists():
        print('[WARN] No classifier checkpoint found. Attribute success metrics will be blank.')
        return None, 'cpu', [CLASSIFIER_THRESHOLD] * len(ATTR_COLUMNS)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = build_attr_classifier()
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device).eval()
    thresholds = ckpt.get('thresholds', [CLASSIFIER_THRESHOLD] * len(ATTR_COLUMNS))
    print('[OK] loaded classifier:', path, 'thresholds:', thresholds)
    return model, device, thresholds

@torch.no_grad()
def predict_attrs(model, device, img: Image.Image):
    if model is None:
        return None
    tf = transforms.Compose([
        transforms.Resize((CLASSIFIER_IMAGE_SIZE, CLASSIFIER_IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    x = tf(img.convert('RGB')).unsqueeze(0).to(device)
    return torch.sigmoid(model(x))[0].detach().cpu().numpy()

In [ ]:
# 7. Evaluate only the two-model comparison set under RUN_DIR/edited

def resolve_meta_path(stored_path: str, meta_path: Path) -> Path:
    p = Path(stored_path)
    if p.exists():
        return p
    if 'processed' in p.parts:
        idx = p.parts.index('processed')
        local = PROCESSED_DIR / Path(*p.parts[idx + 1:])
        if local.exists():
            return local
    local = meta_path.parent / p.name
    if local.exists():
        return local
    return p


def evaluate_edits():
    model, device, thresholds = load_classifier(CLASSIFIER_PATH)
    all_meta_files = sorted((RUN_DIR / 'edited').glob('*/*/*_meta.json'))
    meta_files = [p for p in all_meta_files if p.parent.parent.name in COMPARE_MODELS]
    ignored = len(all_meta_files) - len(meta_files)
    print('Meta files used:', len(meta_files), '| ignored non-compare models:', ignored)
    rows = []
    for meta_path in tqdm(meta_files, desc='evaluate'):
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        if meta.get('model_id') not in COMPARE_MODELS:
            continue
        task = meta['task']
        idx = TASK_TO_ATTR_INDEX[task]
        orig_path = resolve_meta_path(meta['source_image'], meta_path)
        edited_path = resolve_meta_path(meta['best_path'], meta_path)
        mask_path = resolve_meta_path(meta.get('mask_soft', meta.get('mask_hard')), meta_path)

        orig = Image.open(orig_path).convert('RGB')
        edited = Image.open(edited_path).convert('RGB').resize(orig.size, Image.Resampling.LANCZOS)
        mask = Image.open(mask_path).convert('L').resize(orig.size, Image.Resampling.BILINEAR)

        p_before = predict_attrs(model, device, orig)
        p_after = predict_attrs(model, device, edited)
        if p_after is None:
            target = 0.0 if task == 'make_older' else 1.0
            attr_success = None
            attr_score = None
            before_prob = None
            after_prob = None
        elif task == 'make_older':
            target = 0.0
            before_prob = float(p_before[idx])
            after_prob = float(p_after[idx])
            attr_success = bool(p_after[idx] < thresholds[idx])
            attr_score = 1.0 - float(p_after[idx])
        else:
            target = 1.0
            before_prob = float(p_before[idx])
            after_prob = float(p_after[idx])
            attr_success = bool(p_after[idx] >= thresholds[idx])
            attr_score = float(p_after[idx])

        selected = meta['candidates'][meta['selected_candidate']]
        rows.append({
            'model_id': meta['model_id'],
            'engine': meta.get('engine', 'inpainting'),
            'task': task,
            'image_id': str(meta['id']),
            'candidate_id': meta['selected_candidate'],
            'selection_score': meta.get('selection_score', selected.get('selection_score')),
            'attr_target': target,
            'attr_prob_before': before_prob,
            'attr_prob_after': after_prob,
            'attr_score': attr_score,
            'attr_success': attr_success,
            'identity_similarity': selected.get('identity_similarity'),
            'lpips': lpips_distance(orig, edited),
            'background_l1': background_l1(orig, edited, mask),
            'background_ssim': background_ssim(orig, edited, mask),
            'strength': meta['strength'],
            'guidance_scale': meta['guidance_scale'],
            'seed': selected['seed'],
            'original_path': str(orig_path),
            'edited_path': str(edited_path),
        })

    results_long = pd.DataFrame(rows)
    metrics_dir = RUN_DIR / 'metrics'
    metrics_dir.mkdir(parents=True, exist_ok=True)
    long_path = metrics_dir / 'results_long.csv'
    results_long.to_csv(long_path, index=False)
    print('[SAVED]', long_path, len(results_long), 'rows')

    if len(results_long):
        agg = {
            'n': ('image_id', 'count'),
            'attr_success_rate': ('attr_success', 'mean'),
            'attr_prob_after_mean': ('attr_prob_after', 'mean'),
            'identity_similarity_mean': ('identity_similarity', 'mean'),
            'lpips_mean': ('lpips', 'mean'),
            'background_l1_mean': ('background_l1', 'mean'),
            'background_ssim_mean': ('background_ssim', 'mean'),
            'selection_score_mean': ('selection_score', 'mean'),
        }
        results_summary = results_long.groupby(['model_id', 'engine', 'task'], as_index=False).agg(**agg)
    else:
        results_summary = pd.DataFrame()
    summary_path = metrics_dir / 'results_summary.csv'
    results_summary.to_csv(summary_path, index=False)
    print('[SAVED]', summary_path)
    display(results_summary)
    return results_long, results_summary

results_long, results_summary = evaluate_edits()

In [ ]:
# 8. Default visuals, pairwise sd15-vs-sdxl visuals, ranking, health check, and export zip
PAIRWISE_DIR = RUN_DIR / 'reports' / 'pairwise'
PAIRWISE_DIR.mkdir(parents=True, exist_ok=True)
(EXPORTS_DIR / 'comparison_grids').mkdir(parents=True, exist_ok=True)


def _open_thumb(path: str, size: int = 256) -> Image.Image:
    return Image.open(path).convert('RGB').resize((size, size), Image.Resampling.LANCZOS)


def _draw_centered_text(draw: ImageDraw.ImageDraw, xy, text: str, fill=(0, 0, 0)):
    draw.text(xy, text, fill=fill)


def make_grid_for_task(results_long: pd.DataFrame, task: str, max_rows: int = 4):
    # Existing/default style: original | edited, with rows across evaluated models.
    sub = results_long[results_long['task'] == task].head(max_rows)
    if sub.empty:
        return None
    thumb = 256
    rows = []
    for _, row in sub.iterrows():
        orig = _open_thumb(row['original_path'], thumb)
        edit = _open_thumb(row['edited_path'], thumb)
        canvas = Image.new('RGB', (thumb * 2, thumb + 28), 'white')
        canvas.paste(orig, (0, 28))
        canvas.paste(edit, (thumb, 28))
        draw = ImageDraw.Draw(canvas)
        draw.text((8, 6), f"{row['model_id']} / {row['image_id']}", fill=(0, 0, 0))
        rows.append(canvas)
    grid = Image.new('RGB', (thumb * 2, len(rows) * (thumb + 28)), 'white')
    for i, row_img in enumerate(rows):
        grid.paste(row_img, (0, i * (thumb + 28)))
    out = RUN_DIR / 'reports' / 'qualitative' / f'qualitative_grid_{task}.png'
    out.parent.mkdir(parents=True, exist_ok=True)
    grid.save(out)
    print('[SAVED]', out)
    return out


def meta_for(model_id: str, task: str, image_id: str) -> Optional[Dict[str, Any]]:
    p = RUN_DIR / 'edited' / model_id / task / f'{image_id}_meta.json'
    if not p.exists():
        return None
    return json.loads(p.read_text(encoding='utf-8'))


def candidate_paths(meta: Optional[Dict[str, Any]]) -> str:
    if not meta:
        return ''
    paths = []
    for c in meta.get('candidates', []):
        paths.append(c.get('blend_path') or c.get('path') or '')
    return ';'.join([p for p in paths if p])


def make_pairwise_grid_for_task(results_long: pd.DataFrame, task: str, max_rows: int = 12):
    # New comparison style: original | sd15 best | sdxl best.
    task_df = results_long[(results_long['task'] == task) & (results_long['model_id'].isin(COMPARE_MODELS))]
    by_model = {m: task_df[task_df['model_id'] == m].set_index('image_id') for m in COMPARE_MODELS}
    id_sets = [set(df.index.astype(str)) for df in by_model.values()]
    if not id_sets:
        return None, []
    common_ids = sorted(set.intersection(*id_sets)) if all(id_sets) else []
    ids = common_ids or sorted(set.union(*id_sets))
    ids = ids[:max_rows]
    if not ids:
        return None, []

    thumb = 256
    header_h = 34
    row_h = thumb + 34
    cols = ['original'] + COMPARE_MODELS
    grid = Image.new('RGB', (thumb * len(cols), header_h + row_h * len(ids)), 'white')
    draw = ImageDraw.Draw(grid)
    for ci, label in enumerate(cols):
        draw.text((ci * thumb + 8, 10), label, fill=(0, 0, 0))

    index_rows = []
    for ri, image_id in enumerate(ids):
        y = header_h + ri * row_h
        first_row = None
        for m in COMPARE_MODELS:
            if image_id in by_model[m].index:
                first_row = by_model[m].loc[image_id]
                if isinstance(first_row, pd.DataFrame):
                    first_row = first_row.iloc[0]
                break
        if first_row is None:
            continue
        orig = _open_thumb(first_row['original_path'], thumb)
        grid.paste(orig, (0, y + 24))
        draw.text((8, y + 4), f'{task} / {image_id}', fill=(0, 0, 0))

        row_out = {'task': task, 'image_id': image_id, 'original_path': first_row['original_path']}
        for ci, model_id in enumerate(COMPARE_MODELS, start=1):
            if image_id in by_model[model_id].index:
                row = by_model[model_id].loc[image_id]
                if isinstance(row, pd.DataFrame):
                    row = row.iloc[0]
                edited = _open_thumb(row['edited_path'], thumb)
                grid.paste(edited, (ci * thumb, y + 24))
                draw.text((ci * thumb + 8, y + 4), f"{model_id} best", fill=(0, 0, 0))
                row_out[f'{model_id}_best_path'] = row['edited_path']
                row_out[f'{model_id}_selected_candidate'] = row['candidate_id']
                row_out[f'{model_id}_selection_score'] = row['selection_score']
                row_out[f'{model_id}_candidates'] = candidate_paths(meta_for(model_id, task, image_id))
            else:
                draw.rectangle([ci * thumb, y + 24, (ci + 1) * thumb - 1, y + 24 + thumb - 1], outline=(180, 180, 180))
                draw.text((ci * thumb + 8, y + 4), f'{model_id} missing', fill=(160, 0, 0))
                row_out[f'{model_id}_best_path'] = ''
                row_out[f'{model_id}_selected_candidate'] = ''
                row_out[f'{model_id}_selection_score'] = ''
                row_out[f'{model_id}_candidates'] = ''
        row_out['chosen_model'] = ''
        row_out['chosen_candidate'] = ''
        row_out['notes'] = ''
        index_rows.append(row_out)

    out = PAIRWISE_DIR / f'pairwise_{COMPARE_MODELS[0]}_vs_{COMPARE_MODELS[1]}_{task}.png'
    grid.save(out)
    shutil.copy2(out, EXPORTS_DIR / 'comparison_grids' / out.name)
    print('[SAVED]', out)
    return out, index_rows


qual_paths = [make_grid_for_task(results_long, task) for task in TASKS]
qual_paths = [p for p in qual_paths if p]

pairwise_paths = []
pairwise_rows = []
for task in TASKS:
    p, rows = make_pairwise_grid_for_task(results_long, task)
    if p:
        pairwise_paths.append(p)
    pairwise_rows.extend(rows)

# Also create one stacked all-task figure for quick visual scanning.
if pairwise_paths:
    imgs = [Image.open(p).convert('RGB') for p in pairwise_paths]
    width = max(i.width for i in imgs)
    height = sum(i.height for i in imgs)
    all_grid = Image.new('RGB', (width, height), 'white')
    y = 0
    for img in imgs:
        all_grid.paste(img, (0, y))
        y += img.height
    all_path = PAIRWISE_DIR / f'pairwise_{COMPARE_MODELS[0]}_vs_{COMPARE_MODELS[1]}_all_tasks.png'
    all_grid.save(all_path)
    shutil.copy2(all_path, EXPORTS_DIR / 'comparison_grids' / all_path.name)
    pairwise_paths.append(all_path)
    print('[SAVED]', all_path)

selection_template = pd.DataFrame(pairwise_rows)
selection_template_path = EXPORTS_DIR / 'human_selection_template.csv'
selection_template.to_csv(selection_template_path, index=False)
print('[SAVED]', selection_template_path)

# Ranking only over the two compare models. Missing models remain visible as pending.
score_by_model = {}
if len(results_summary):
    base_rank = results_summary[results_summary['model_id'].isin(COMPARE_MODELS)].groupby('model_id', as_index=False).agg(
        attr_success_rate=('attr_success_rate', 'mean'),
        lpips_mean=('lpips_mean', 'mean'),
        background_l1_mean=('background_l1_mean', 'mean'),
        background_ssim_mean=('background_ssim_mean', 'mean'),
        selection_score_mean=('selection_score_mean', 'mean'),
    )
    if len(base_rank):
        base_rank['overall_score'] = (
            base_rank['attr_success_rate'].fillna(0) * 0.45
            + (1 - base_rank['lpips_mean'].fillna(0.35).clip(0, 0.35) / 0.35) * 0.15
            + (1 - base_rank['background_l1_mean'].fillna(0.08).clip(0, 0.08) / 0.08) * 0.15
            + base_rank['background_ssim_mean'].fillna(0).clip(0, 1) * 0.15
            + base_rank['selection_score_mean'].fillna(0) * 0.10
        )
        score_by_model = {r['model_id']: r.to_dict() for _, r in base_rank.iterrows()}

ranking_rows = []
for model_id in COMPARE_MODELS:
    row = score_by_model.get(model_id)
    if row:
        row = dict(row)
        row['status'] = 'evaluated'
    else:
        row = {'model_id': model_id, 'status': 'pending', 'overall_score': np.nan}
    ranking_rows.append(row)
rank = pd.DataFrame(ranking_rows).sort_values(['status', 'overall_score'], ascending=[True, False])

# Task/model comparison table for report and export.
if len(results_summary):
    comparison_table = results_summary[results_summary['model_id'].isin(COMPARE_MODELS)].copy()
else:
    comparison_table = pd.DataFrame()

health_rows = []
from safetensors import safe_open
for model_id in COMPARE_MODELS:
    lora = LORA_DIR / f'{model_id}_faceattr_lora.safetensors'
    tensors = None
    ok = False
    if lora.exists():
        try:
            with safe_open(lora, framework='pt', device='cpu') as f:
                tensors = len(list(f.keys()))
            ok = True
        except Exception:
            ok = False
    health_rows.append({
        'model_id': model_id,
        'lora_exists': lora.exists(),
        'lora_bytes': lora.stat().st_size if lora.exists() else 0,
        'safetensors_ok': ok,
        'tensor_count': tensors,
        'edited_meta_total': len(list((RUN_DIR / 'edited' / model_id).glob('*/*_meta.json'))),
    })
health = pd.DataFrame(health_rows)

for src in [RUN_DIR/'metrics'/'results_long.csv', RUN_DIR/'metrics'/'results_summary.csv']:
    if src.exists():
        shutil.copy2(src, EXPORTS_DIR / src.name)
rank.to_csv(EXPORTS_DIR / 'ranking.csv', index=False)
health.to_csv(EXPORTS_DIR / 'model_health_check.csv', index=False)
comparison_table.to_csv(EXPORTS_DIR / 'comparison_table.csv', index=False)

report = RUN_DIR / 'reports' / 'final_report.md'
lines = [
    '# Face Attribute Editing Evaluation Report - SD1.5 vs SDXL',
    '',
    f'- Created: {time.strftime("%Y-%m-%d %H:%M:%S")}',
    f'- Compare models: {COMPARE_MODELS}',
    f'- Models run this session: {MODELS_TO_RUN}',
    f'- Split: {SPLIT_NAME}',
    f'- Samples per task: {SAMPLES_PER_TASK}',
    f'- Candidates per image: {NUM_CANDIDATES}',
    '',
    '## Results Summary',
    '',
    comparison_table.to_markdown(index=False) if len(comparison_table) else '_No rows_',
    '',
    '## Ranking',
    '',
    rank.to_markdown(index=False) if len(rank) else '_No ranking_',
    '',
    '## Pairwise Visuals',
    '',
    '\n'.join(f'- `{p.relative_to(RUN_DIR)}`' for p in pairwise_paths) if pairwise_paths else '_No pairwise grids_',
    '',
    '## Human Selection Template',
    '',
    f'- `{selection_template_path}`',
    '',
    '## Health Check',
    '',
    health.to_markdown(index=False),
]
report.write_text('\n'.join(lines), encoding='utf-8')
shutil.copy2(report, EXPORTS_DIR / 'final_report.md')

zip_base = DRIVE_PROJECT / 'face_attr_edit_eval_outputs_sd15_sdxl'
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=str(EXPORTS_DIR))
print('[ZIP]', zip_path)
display(rank)
display(health)
display(selection_template.head())